In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from pathlib import Path

In [2]:
import numpy as np
print(np.__version__)

2.4.6


In [3]:
import shap

c:\Users\Nithu\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:



base_path = Path("D:/Nithyashri/Hackathons/idbi/dataset/home-credit-default-risk")

app_test = pd.read_csv(base_path / "application_test.csv")
app_train = pd.read_csv(base_path / "application_train.csv")

In [6]:
print("Train shape:", app_train.shape)
print("Test shape:", app_test.shape)

# 1. Check target distribution (class imbalance)
print("\nTarget distribution:")
print(app_train['TARGET'].value_counts(normalize=True))


Train shape: (307511, 122)
Test shape: (48744, 121)

Target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [7]:
# 2. Check missing values - top 20 columns by % missing
missing = app_train.isnull().mean().sort_values(ascending=False) * 100
print("\nTop 20 columns by missing %:")
print(missing.head(20))




Top 20 columns by missing %:
COMMONAREA_AVG              69.872297
COMMONAREA_MODE             69.872297
COMMONAREA_MEDI             69.872297
NONLIVINGAPARTMENTS_MEDI    69.432963
NONLIVINGAPARTMENTS_MODE    69.432963
NONLIVINGAPARTMENTS_AVG     69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_AVG        68.354953
LIVINGAPARTMENTS_MEDI       68.354953
LIVINGAPARTMENTS_MODE       68.354953
FLOORSMIN_MODE              67.848630
FLOORSMIN_AVG               67.848630
FLOORSMIN_MEDI              67.848630
YEARS_BUILD_AVG             66.497784
YEARS_BUILD_MODE            66.497784
YEARS_BUILD_MEDI            66.497784
OWN_CAR_AGE                 65.990810
LANDAREA_MEDI               59.376738
LANDAREA_AVG                59.376738
LANDAREA_MODE               59.376738
dtype: float64


In [8]:
# 3. Data types overview
print("\nData types:")
print(app_train.dtypes.value_counts())


Data types:
float64    65
int64      41
str        16
Name: count, dtype: int64


In [9]:
# 4. Quick look at numeric vs categorical columns
numeric_cols = app_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = app_train.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")


Numeric columns: 106
Categorical columns: 16


C:\Users\Nithu\AppData\Local\Temp\ipykernel_39824\1968515300.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = app_train.select_dtypes(include=['object']).columns.tolist()


In [10]:
# 5. Check for anomalies and fix DAYS_EMPLOYED placeholder

# DAYS_EMPLOYED has a placeholder value of 365243 for "not employed"
anomaly_count = (app_train['DAYS_EMPLOYED'] == 365243).sum()
print(f"\nDAYS_EMPLOYED anomaly (365243 placeholder): {anomaly_count} rows")

# Add anomaly flag column
app_train['DAYS_EMPLOYED_ANOM'] = app_train['DAYS_EMPLOYED'] == 365243
app_test['DAYS_EMPLOYED_ANOM'] = app_test['DAYS_EMPLOYED'] == 365243

# Replace anomaly with NaN
app_train['DAYS_EMPLOYED'] = app_train['DAYS_EMPLOYED'].replace(365243, np.nan)
app_test['DAYS_EMPLOYED'] = app_test['DAYS_EMPLOYED'].replace(365243, np.nan)


DAYS_EMPLOYED anomaly (365243 placeholder): 55374 rows


C:\Users\Nithu\AppData\Local\Temp\ipykernel_39824\845093647.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_train['DAYS_EMPLOYED_ANOM'] = app_train['DAYS_EMPLOYED'] == 365243
C:\Users\Nithu\AppData\Local\Temp\ipykernel_39824\845093647.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_test['DAYS_EMPLOYED_ANOM'] = app_test['DAYS_EMPLOYED'] == 365243


In [12]:
# 6. Sanity check on key columns for ratio features
print("\nKey column summary:")
print(app_train[['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_BIRTH']].describe())


Key column summary:
       AMT_INCOME_TOTAL    AMT_CREDIT    AMT_ANNUITY     DAYS_BIRTH
count      3.075110e+05  3.075110e+05  307499.000000  307511.000000
mean       1.687979e+05  5.990260e+05   27108.573909  -16036.995067
std        2.371231e+05  4.024908e+05   14493.737315    4363.988632
min        2.565000e+04  4.500000e+04    1615.500000  -25229.000000
25%        1.125000e+05  2.700000e+05   16524.000000  -19682.000000
50%        1.471500e+05  5.135310e+05   24903.000000  -15750.000000
75%        2.025000e+05  8.086500e+05   34596.000000  -12413.000000
max        1.170000e+08  4.050000e+06  258025.500000   -7489.000000


In [11]:
# 7. Load the bureau and bureau_balance datasets for further analysis

import pandas as pd
import numpy as np

bureau = pd.read_csv(base_path / "bureau.csv")
bureau_balance = pd.read_csv(base_path / "bureau_balance.csv")

print("Bureau shape:", bureau.shape)
print("Bureau balance shape:", bureau_balance.shape)
print("\nBureau columns:", bureau.columns.tolist())

Bureau shape: (1716428, 17)
Bureau balance shape: (27299925, 3)

Bureau columns: ['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']


In [13]:
# 8. Convert bureau_balance to categorical and check value counts

bureau_balance['STATUS_NUM'] = bureau_balance['STATUS'].replace({'C': 0, 'X': 0}).astype(int)
print("Value counts for STATUS_NUM:")
print(bureau_balance['STATUS_NUM'].value_counts())

Value counts for STATUS_NUM:
STATUS_NUM
0    26956982
1      242347
5       62406
2       23419
3        8924
4        5847
Name: count, dtype: int64


In [14]:
# 9. Aggregate bureau balance data by bureau loan ID

bb_agg = bureau_balance.groupby('SK_ID_BUREAU').agg(
    BB_MONTHS_COUNT=('MONTHS_BALANCE', 'count'),
    BB_MAX_STATUS=('STATUS_NUM', 'max'),
    BB_MEAN_STATUS=('STATUS_NUM', 'mean'),
    BB_LATE_MONTHS=('STATUS_NUM', lambda x: (x > 0).sum())
).reset_index()

print("\nBureau balance aggregated shape:", bb_agg.shape)


Bureau balance aggregated shape: (817395, 5)


In [15]:
#10. Merge the aggregated bureau balance info into the bureau table

bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
print("\nBureau shape after merging aggregated bureau balance info:", bureau.shape)


Bureau shape after merging aggregated bureau balance info: (1716428, 21)


In [16]:
# 11 Aggregate bureau data up to customer level

bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_LOAN_COUNT=('SK_ID_BUREAU', 'count'),
    BUREAU_ACTIVE_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    BUREAU_CLOSED_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),
    BUREAU_CREDIT_SUM=('AMT_CREDIT_SUM', 'sum'),
    BUREAU_CREDIT_SUM_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_CREDIT_SUM_OVERDUE=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    BUREAU_MAX_OVERDUE=('AMT_CREDIT_MAX_OVERDUE', 'max'),
    BUREAU_DAYS_CREDIT_MEAN=('DAYS_CREDIT', 'mean'),
    BUREAU_DAYS_CREDIT_MIN=('DAYS_CREDIT', 'min'),
    BUREAU_LATE_MONTHS_SUM=('BB_LATE_MONTHS', 'sum'),
    BUREAU_MAX_STATUS_EVER=('BB_MAX_STATUS', 'max'),
).reset_index()

In [17]:
# 12 Create one derived ratio feature
bureau_agg['BUREAU_CREDIT_TO_DEBT_RATIO'] = bureau_agg['BUREAU_CREDIT_SUM'] / (bureau_agg['BUREAU_CREDIT_SUM_DEBT'] + 1)

In [18]:
#13 Show results and save the file

print("\nAggregated Bureau Data:")
print(bureau_agg.head())

bureau_agg.to_csv("bureau_agg.csv", index=False)


Aggregated Bureau Data:
   SK_ID_CURR  BUREAU_LOAN_COUNT  BUREAU_ACTIVE_COUNT  BUREAU_CLOSED_COUNT  \
0      100001                  7                    3                    4   
1      100002                  8                    2                    6   
2      100003                  4                    1                    3   
3      100004                  2                    0                    2   
4      100005                  3                    2                    1   

   BUREAU_CREDIT_SUM  BUREAU_CREDIT_SUM_DEBT  BUREAU_CREDIT_SUM_OVERDUE  \
0        1453365.000                596686.5                        0.0   
1         865055.565                245781.0                        0.0   
2        1017400.500                     0.0                        0.0   
3         189037.800                     0.0                        0.0   
4         657126.000                568408.5                        0.0   

   BUREAU_MAX_OVERDUE  BUREAU_DAYS_CREDIT_MEAN  BUREAU_

In [23]:
# Load previous applications
prev_app = pd.read_csv(base_path / 'previous_application.csv')


print("Previous application shape:", prev_app.shape)
print("\nColumns:", prev_app.columns.tolist())

# Check the approval status categories
print("\nNAME_CONTRACT_STATUS values:")
print(prev_app['NAME_CONTRACT_STATUS'].value_counts())

Previous application shape: (1670214, 37)

Columns: ['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY', 'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'NAME_CASH_LOAN_PURPOSE', 'NAME_CONTRACT_STATUS', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'SELLERPLACE_AREA', 'NAME_SELLER_INDUSTRY', 'CNT_PAYMENT', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION', 'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION', 'NFLAG_INSURED_ON_APPROVAL']

NAME_CONTRACT_STATUS values:
NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count

In [26]:


# ---- Aggregate previous_application.csv up to SK_ID_CURR ----
prev_agg = prev_app.groupby('SK_ID_CURR').agg(
    PREV_APP_COUNT=('SK_ID_PREV', 'count'),
    PREV_APPROVED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    PREV_REFUSED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    PREV_CANCELED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Canceled').sum()),
    PREV_AMT_APPLICATION_MEAN=('AMT_APPLICATION', 'mean'),
    PREV_AMT_CREDIT_MEAN=('AMT_CREDIT', 'mean'),
    PREV_AMT_ANNUITY_MEAN=('AMT_ANNUITY', 'mean'),
    PREV_AMT_DOWN_PAYMENT_MEAN=('AMT_DOWN_PAYMENT', 'mean'),
    PREV_DAYS_DECISION_MEAN=('DAYS_DECISION', 'mean'),
    PREV_DAYS_DECISION_MIN=('DAYS_DECISION', 'min'),  # most recent decision (closest to 0)
    PREV_CNT_PAYMENT_MEAN=('CNT_PAYMENT', 'mean'),
).reset_index()



In [ ]:
# Derived ratio: how often were they refused historically
prev_agg['PREV_REFUSAL_RATIO'] = (
    prev_agg['PREV_REFUSED_COUNT'] / prev_agg['PREV_APP_COUNT'].replace(0, np.nan)
)



In [27]:
# Derived ratio: approval ratio
prev_agg['PREV_APPROVAL_RATIO'] = (
    prev_agg['PREV_APPROVED_COUNT'] / prev_agg['PREV_APP_COUNT'].replace(0, np.nan)
 )

print("\nFinal prev_agg shape:", prev_agg.shape)
print(prev_agg.head())

# Save (ensure directory exists in base_path)
processed_dir = base_path / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
prev_agg.to_csv(processed_dir / 'prev_agg.csv', index=False)


Final prev_agg shape: (338857, 13)
   SK_ID_CURR  PREV_APP_COUNT  PREV_APPROVED_COUNT  PREV_REFUSED_COUNT  \
0      100001               1                    1                   0   
1      100002               1                    1                   0   
2      100003               3                    3                   0   
3      100004               1                    1                   0   
4      100005               2                    1                   0   

   PREV_CANCELED_COUNT  PREV_AMT_APPLICATION_MEAN  PREV_AMT_CREDIT_MEAN  \
0                    0                   24835.50              23787.00   
1                    0                  179055.00             179055.00   
2                    0                  435436.50             484191.00   
3                    0                   24282.00              20106.00   
4                    1                   22308.75              20076.75   

   PREV_AMT_ANNUITY_MEAN  PREV_AMT_DOWN_PAYMENT_MEAN  PREV_DAYS_DECI

In [29]:
# Load installments
installments = pd.read_csv(base_path / 'installments_payments.csv')

# Derived columns first
installments['DAYS_LATE'] = installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']
installments['PAYMENT_DIFF'] = installments['AMT_INSTALMENT'] - installments['AMT_PAYMENT']  # positive = underpaid

# Aggregate to one row per customer
install_agg = installments.groupby('SK_ID_CURR').agg(
    INSTALL_COUNT=('SK_ID_PREV', 'count'),
    INSTALL_DAYS_LATE_MEAN=('DAYS_LATE', 'mean'),
    INSTALL_DAYS_LATE_MAX=('DAYS_LATE', 'max'),
    INSTALL_LATE_COUNT=('DAYS_LATE', lambda x: (x > 0).sum()),
    INSTALL_PAYMENT_DIFF_MEAN=('PAYMENT_DIFF', 'mean'),
    INSTALL_UNDERPAID_COUNT=('PAYMENT_DIFF', lambda x: (x > 0).sum()),
).reset_index()

install_agg['INSTALL_LATE_RATIO'] = install_agg['INSTALL_LATE_COUNT'] / install_agg['INSTALL_COUNT']

# Save (ensure directory exists in base_path)
processed_dir = base_path / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
install_agg.to_csv(processed_dir / 'install_agg.csv', index=False)
print(install_agg.shape)

(339587, 8)


In [31]:
pos_cash = pd.read_csv( base_path / 'POS_CASH_balance.csv')

pos_agg = pos_cash.groupby('SK_ID_CURR').agg(
    POS_COUNT=('SK_ID_PREV', 'count'),
    POS_CNT_INSTALMENT_MEAN=('CNT_INSTALMENT', 'mean'),
    POS_CNT_INSTALMENT_FUTURE_MEAN=('CNT_INSTALMENT_FUTURE', 'mean'),
    POS_DPD_MEAN=('SK_DPD', 'mean'),
    POS_DPD_MAX=('SK_DPD', 'max'),
    POS_DPD_DEF_MEAN=('SK_DPD_DEF', 'mean'),
    POS_LATE_COUNT=('SK_DPD', lambda x: (x > 0).sum()),
).reset_index()

pos_agg['POS_LATE_RATIO'] = pos_agg['POS_LATE_COUNT'] / pos_agg['POS_COUNT']

processed_dir = base_path / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
pos_agg.to_csv(processed_dir / 'pos_agg.csv', index=False)
print(pos_agg.shape)

(337252, 9)


In [33]:
# Start from the main table
master = app_train.copy()

# Left-join every aggregated table on SK_ID_CURR
master = master.merge(bureau_agg, on='SK_ID_CURR', how='left')
master = master.merge(prev_agg, on='SK_ID_CURR', how='left')
master = master.merge(install_agg, on='SK_ID_CURR', how='left')
master = master.merge(pos_agg, on='SK_ID_CURR', how='left')

print(master.shape)

# Save it - this is your final modeling table
master.to_csv(processed_dir / 'master_train.csv', index=False)

(307511, 162)
